In [1]:
from fastmcp import Client
 
async def main():
    # 通过 stdio 连接到本地脚本
    async with Client(r".\tools\launch_app.py") as client:
        tools = await client.list_tools()
        print(f"可用工具：{tools}")
        result = await client.call_tool("add", {"a": 5, "b": 3})
        print(f"结果：{result.text}")
 
    # # 通过 SSE 连接
    # async with Client("http://localhost:8000/sse") as client:
    #     # ... 使用客户端
    #     pass

In [2]:
import asyncio
await main()

RuntimeError: Client failed to connect: fileno

In [ ]:
import winreg
import os


{'TTime': '"C:\\Program Files\\TTime\\Uninstall TTime.exe" /allusers', 'Octopus 8.7.7': '"C:\\Program Files\\Octopus\\Uninstall Octopus.exe" /allusers', 'cool-clock 1.1.0': '"C:\\Program Files\\cool-clock\\Uninstall cool-clock.exe" /allusers', 'DbVisualizer': 'C:\\Program Files\\DbVisualizer', 'Android Studio': 'C:\\Program Files\\Android\\Android Studio\\uninstall.exe', 'Armoury Crate Service': '"C:\\Program Files\\ASUS\\Armoury Crate Service\\ArmouryCrate.Uninstaller.exe"', 'Bandizip': 'C:\\Program Files\\Bandizip\\', 'Cisco Packet Tracer 7.0 64Bit': 'C:\\Program Files\\Cisco Packet Tracer 7.0\\', 'Clash Verge': '"C:\\Program Files\\Clash Verge"', 'Docker Desktop': 'C:\\Program Files\\Docker\\Docker', '有道云笔记 8.2.10': '"C:\\Program Files\\ynote-desktop\\Uninstall 有道云笔记.exe" /allusers', 'MiniMax 3.0.5': '"C:\\Program Files\\MiniMax\\Uninstall MiniMax.exe" /allusers', 'Boris FX Sapphire Plug-ins for Photoshop': 'C:\\Program Files\\BorisFX\\Sapphire 2024.5 Photoshop\\', 'Git': 'C:\\Progr

In [9]:
from pathlib import Path
import os
from fuzzywuzzy import fuzz
import re
import pypinyin  # 需要安装: pip install pypinyin
import winreg  # 需要在代码开头导入

def get_installed_apps():
    """获取 Windows 已安装的应用"""
    apps = {}
    # 注册表路径
    paths = [
        r"SOFTWARE\Microsoft\Windows\CurrentVersion\Uninstall",
        r"SOFTWARE\WOW6432Node\Microsoft\Windows\CurrentVersion\Uninstall"
    ]
    
    for path in paths:
        try:
            with winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE, path) as key:
                for i in range(winreg.QueryInfoKey(key)[0]):
                    try:
                        subkey_name = winreg.EnumKey(key, i)
                        with winreg.OpenKey(key, subkey_name) as subkey:
                            app_name = winreg.QueryValueEx(subkey, "DisplayName")[0]
                            # 尝试获取安装路径或执行文件
                            try:
                                install_loc = winreg.QueryValueEx(subkey, "InstallLocation")[0]
                            except FileNotFoundError:
                                try:
                                    install_loc = winreg.QueryValueEx(subkey, "UninstallString")[0]
                                except:
                                    install_loc = None
                            
                            if app_name and install_loc:
                                apps[app_name] = install_loc
                    except:
                        continue
        except:
            continue
    return apps

def chinese_to_pinyin(text):
    """
    将中文转换为拼音，去除声调
    """
    try:
        # 转换为拼音并连接成字符串
        pinyin_list = pypinyin.lazy_pinyin(text, style=pypinyin.NORMAL)
        return ''.join(pinyin_list).lower()
    except:
        return text.lower()

def clean_application_name(name):
    """
    清理应用程序名称，移除版本号等干扰信息，并支持中文拼音转换
    """
    # 移除版本号
    name = re.sub(r'\s+v?\d+[\d\.]*\s*', ' ', name)
    name = re.sub(r'\s+\d+\.\d+.*', ' ', name)
    # 移除括号中的内容（如版本信息）
    name = re.sub(r'\([^)]*\)', '', name)
    name = re.sub(r'\[[^\]]*\]', '', name)
    # 转换为小写并清理空格
    cleaned = name.strip().lower()
    
    # 返回原始名称及其拼音形式
    original = cleaned
    pinyin_version = chinese_to_pinyin(cleaned)
    
    return original, pinyin_version

def calculate_similarity_score(clean_app_name, exe_name):
    """
    计算综合相似度分数，考虑中文、拼音、英文的混合匹配
    """
    # 清理EXE名称
    exe_name_clean = re.sub(r'[_\-]', ' ', exe_name.lower())
    
    # 获取应用名称的原始和拼音版本
    app_original, app_pinyin = clean_app_name
    
    # 计算各种匹配分数
    scores = []
    
    # 原始名称匹配
    scores.append(fuzz.ratio(app_original, exe_name_clean))
    scores.append(fuzz.partial_ratio(app_original, exe_name_clean))
    scores.append(fuzz.token_set_ratio(app_original, exe_name_clean))
    
    # 拼音匹配
    scores.append(fuzz.ratio(app_pinyin, exe_name_clean))
    scores.append(fuzz.partial_ratio(app_pinyin, exe_name_clean))
    scores.append(fuzz.token_set_ratio(app_pinyin, exe_name_clean))
    
    # EXE名称转拼音后的匹配
    exe_pinyin = chinese_to_pinyin(exe_name)
    scores.append(fuzz.ratio(app_pinyin, exe_pinyin))
    scores.append(fuzz.partial_ratio(app_pinyin, exe_pinyin))
    scores.append(fuzz.token_set_ratio(app_pinyin, exe_pinyin))
    
    # 特殊处理：检查是否包含关键字符
    if app_pinyin in exe_name_clean or exe_name_clean in app_pinyin:
        scores.append(90)  # 高分
    
    # 检查字符级别相似性
    char_similarity = fuzz.partial_ratio(app_pinyin.replace(' ', ''), exe_name_clean.replace(' ', ''))
    scores.append(char_similarity)
    
    return max(scores)

def contains_main_keywords(clean_app_name, exe_name):
    """
    检查EXE名称是否包含应用名称的主要关键词，支持中文拼音
    """
    app_original, app_pinyin = clean_app_name
    
    # 检查原始名称、拼音是否在exe名称中
    exe_lower = exe_name.lower()
    return (app_original in exe_lower or 
            app_pinyin in exe_lower or
            exe_lower in app_pinyin or
            exe_lower in app_original)

def find_best_exe_match(app_name, exe_files, uninstall_keywords=None):
    """
    根据应用名称在EXE文件列表中进行模糊匹配，找到最佳匹配项
    支持中文、拼音、英文混合匹配
    """
    if uninstall_keywords is None:
        uninstall_keywords = ['uninstall', 'unins000', 'setup', 'install', 'update', 
                             'repair', 'config', 'settings', 'launcher', 'helper', 
                             'patch', 'updater', 'crashpad', 'agent', 'service',
                             'uninst', 'remove', 'wizard', 'utility', 'support']

    # 清理应用名称，获取原始和拼音版本
    clean_app_name = clean_application_name(app_name)
    
    print(f"应用名称: {app_name}")
    print(f"清理后原始名称: {clean_app_name[0]}")
    print(f"拼音: {clean_app_name[1]}")

    best_match = None
    best_score = -1

    for exe_path in exe_files:
        exe_name = exe_path.stem.lower()  # 不带扩展名的文件名

        # 检查是否为卸载程序或不重要的程序
        if any(keyword in exe_name for keyword in uninstall_keywords):
            print(f"  跳过卸载程序: {exe_path.name}")
            continue

        # 计算综合匹配分数
        total_score = calculate_similarity_score(clean_app_name, exe_name)
        
        print(f"  检查文件: {exe_path.name}, 匹配分数: {total_score}")

        # 检查是否包含应用名称的关键部分
        if contains_main_keywords(clean_app_name, exe_name):
            total_score += 20  # 增加权重
            print(f"    包含关键词，加分后分数: {total_score}")

        # 特殊处理：对常见的主程序名称给予额外加分
        main_app_indicators = ['main', 'primary', 'app', 'start', 'run', 'exec']
        if any(indicator in exe_name for indicator in main_app_indicators):
            total_score += 10
        
        # 如果当前应用有特定的指示符（如微信对应weixin）
        app_specific = clean_app_name[1]  # 拼音
        if app_specific in exe_name or exe_name in app_specific:
            total_score += 15
            print(f"    应用特定匹配，加分后分数: {total_score}")

        # 如果得分更高，则更新最佳匹配
        if total_score > best_score:
            best_score = total_score
            best_match = exe_path
            print(f"    新的最佳匹配: {exe_path.name} (分数: {total_score})")

    print(f"最终最佳匹配: {best_match}, 分数: {best_score}")
    
    # 设置最低分数阈值，避免误匹配
    return best_match if best_score >= 20 else None

def improved_exe_finder(app_name, apps_dict):
    """
    改进的EXE文件查找器，支持中文拼音匹配
    """
    if app_name not in apps_dict:
        print(f"应用 {app_name} 未在注册表中找到")
        return None

    install_location = apps_dict[app_name]

    # 处理带引号的路径
    if isinstance(install_location, str) and install_location.startswith('"'):
        install_location = install_location.split('"')[1]

    print(f"安装路径: {install_location}")
    
    path = Path(install_location)

    if path.is_dir():
        # 收集所有EXE文件
        exe_files = list(path.rglob("*.exe"))
        print(f"找到 {len(exe_files)} 个EXE文件")

        if not exe_files:
            print("目录中没有找到EXE文件")
            return None

        # 打印所有找到的EXE文件
        print("所有找到的EXE文件:")
        for exe in exe_files:
            print(f"  - {exe.name}")

        # 使用模糊匹配找到最佳EXE
        best_exe = find_best_exe_match(app_name, exe_files)

        if best_exe:
            # 验证文件是否存在
            if best_exe.exists():
                return best_exe
            else:
                print(f"文件不存在: {best_exe}")

        # 如果没有找到匹配项，列出所有文件及它们的匹配度
        print(f"\n所有文件的匹配度分析:")
        
        # 获取应用名称用于计算匹配度
        clean_app_name = clean_application_name(app_name)
        
        for exe in exe_files:
            score = calculate_similarity_score(clean_app_name, exe.stem.lower())
            status = " [推荐]" if score > 30 else ""
            print(f"  - {exe.name} (匹配度: {score:.1f}%){status}")

    elif path.is_file() and path.suffix.lower() == '.exe':
        # 如果本身就是EXE文件
        return path

    return None

def safe_launch_exe(app_name, apps_dict):
    """
    安全地启动EXE文件，支持中文拼音匹配
    """
    result = improved_exe_finder(app_name, apps_dict)

    if result:
        print(f"\n找到最佳匹配: {result}")
        confirm = input("确认启动? (y/n): ")
        if confirm.lower() == 'y':
            os.startfile(result)
            return result
        else:
            print("已取消")
    else:
        print(f"无法找到 {app_name} 的主程序")
    
    return None



apps = get_installed_apps()
result = safe_launch_exe("vscode", apps)
result
# if not result:
#     print("\n尝试手动指定路径...")
#     # 如果自动查找失败，可以尝试手动指定路径
#     wechat_paths = [
#         r"C:\Program Files\Tencent\Weixin\Weixin.exe",
#         r"C:\Program Files (x86)\Tencent\Weixin\Weixin.exe",
#         Path(apps.get("微信", "")).parent / "Weixin.exe" if "微信" in apps else None
#     ]
    
#     for path in wechat_paths:
#         if path and Path(path).exists():
#             print(f"找到微信路径: {path}")
#             confirm = input("是否启动? (y/n): ")
#             if confirm.lower() == 'y':
#                 os.startfile(path)
#                 break

应用 vscode 未在注册表中找到
无法找到 vscode 的主程序


In [10]:
os.startfile(r"C:\ProgramData\Microsoft\Windows\Start Menu\Programs\Google Chrome.lnk")